Sequential Workflows using LangChain

In [1]:
!pip install langchain langchain-core langchain-community

In [2]:
from langchain_core.prompts import PromptTemplate #helpful for creating individual chains
from langchain_classic.chains import LLMChain, SequentialChain #helpful when we are building the sequential workflow
import os # os is needed because we will create our own virtual environment

In [3]:
!pip install langchain-google-genai

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI #helpful for configuring the LLM - Model

We are building a Sequential Workflow with 4 Chains

- Chain 1 which takes a topic from the user and creates a 'Blog Title' based on the input received from the user

- Chain 2 takes the 'Blog Title' generated in the first chain and creates an 'Outline' for the Blog

- Chain 3 takes the 'Outline' created in the previous chain as input and generates a 'Blog'

- Chain 4 uses the 'Blog' generated in the third chain and creates a 'Summary' of the blog

In [5]:
os.environ["GEMINI_API_KEY"]="API KEY"

In [6]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)

In [7]:
#Chain 1 - Generating a title for the Blog
title_prompt=PromptTemplate(
    input_variables=['topic'], #input from the user based on which the 'Blog Title' needs to be created
    template='Create a short but catchy blog title for the following topic: {topic}'
)

title_chain=LLMChain(
    llm=llm, #gemini-3.6-flash as our LLM
    prompt=title_prompt,
    output_key='blog_title' #this is the expected output from Chain 1
)

/tmp/ipykernel_10381/3736142936.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title_chain=LLMChain(


In [8]:
#Chain 2 - Generating an outline from the title generated
outline_prompt=PromptTemplate(
    input_variables=['blog_title'], #output from Chain 1
    template='Provide a structured outline for this blog title: {blog_title}'
)

outline_chain=LLMChain(
    llm=llm,
    prompt=outline_prompt,
    output_key='blog_outline' #this is the expected output from Chain 2
)

In [9]:
#Chain 3 - Creating a Blog
blog_prompt=PromptTemplate(
    input_variables=['blog_outline'], #output from Chain 2
    template='Draft a blog based on this outline: {blog_outline}')

blog_chain=LLMChain(
    llm=llm,
    prompt=blog_prompt,
    output_key='blog' #this is the expected output from Chain 3
)

In [10]:
#Chain 4 - Summary of the Blog
summary_prompt=PromptTemplate(
    input_variables=['blog'], #output from Chain 3
    template='Summarize this blog in less than 100 words: {blog}'
)

summary_chain=LLMChain(
    llm=llm,
    prompt=summary_prompt,
    output_key='summary' #this is the expected output from Chain 4
)

In [11]:
#Combining the Chains into a Sequential Workflow
overall_chain=SequentialChain(
    chains=[title_chain, outline_chain, blog_chain, summary_chain],
    input_variables=['topic'], #input from the user
    output_variables=['blog_title', 'blog_outline', 'blog', 'summary'], #expected output from all the chains
)

In [12]:
result=overall_chain({"topic":"Role of AI in Banking and Cybersecurity domain"})

/tmp/ipykernel_10381/2539962042.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result=overall_chain({"topic":"Role of AI in Banking and Cybersecurity domain"})


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 12.108431567s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [ ]:
print(result['blog_title'])

In [ ]:
print(result['blog'])